Import necessary libraries first.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

Import the master dataframe with all coins and market features.

In [2]:
master_df = pd.read_csv("market-master-df.csv")

Ensure the data is in chronological order and seperated by symbol.

In [3]:
master_df['open_time'] = pd.to_datetime(master_df['open_time'])
master_df = master_df.sort_values(by=['open_time', 'symbol']).reset_index(drop=True)

In [4]:
print(master_df.head(10))

                  open_time     open     high      low    close      volume  \
0 2017-08-17 04:00:00+00:00  4261.48  4313.62  4261.32  4308.83   47.181009   
1 2017-08-17 04:00:00+00:00   301.13   302.57   298.00   301.61  125.668770   
2 2017-08-17 05:00:00+00:00  4308.83  4328.69  4291.37  4315.32   23.234916   
3 2017-08-17 05:00:00+00:00   301.61   303.28   300.00   303.10  377.672460   
4 2017-08-17 06:00:00+00:00  4330.29  4345.45  4309.37  4324.35    7.229691   
5 2017-08-17 06:00:00+00:00   302.40   304.44   301.90   302.68  303.866720   
6 2017-08-17 07:00:00+00:00  4316.62  4349.99  4287.41  4349.99    4.443249   
7 2017-08-17 07:00:00+00:00   302.68   307.96   302.60   307.96  754.745100   
8 2017-08-17 08:00:00+00:00  4333.32  4377.85  4333.32  4360.69    0.972807   
9 2017-08-17 08:00:00+00:00   307.95   309.97   307.00   308.62  150.750290   

    symbol  log_ret_1d  log_ret_5d  log_ret_21d  ...  log_volume  \
0  BTCUSDT         NaN         NaN          NaN  ...    3.8749

Create the mapping between the string symbols and ints for ML ingestion

In [5]:
# MAP = {
#     'BTCUSDT': 0,
#     'ETHUSDT': 1,
#     'BNBUSDT': 2,
#     'NEOUSDT': 3,
#     'LTCUSDT': 4,
#     'QTUMUSDT': 5,
#     'ADAUSDT': 6,
#     'XRPUSDT': 7,
#     'IOTAUSDT': 8,
#     'XLMUSDT': 9,
#     'TRXUSDT': 10,
#     'ETCUSDT': 11,
#     'VETUSDT': 12,
#     'LINKUSDT': 13,
#     'BATUSDT': 14,
#     'ZECUSDT': 15,
#     'DASHUSDT': 16,
#     'DOGEUSDT': 17,
#     'BCHUSDT': 18,
#     'DGBUSDT': 19,
#     'FILUSDT': 20,
# }

# master_df['symbol'] = master_df['symbol'].map(MAP)

In [6]:
# master_df.head(10)

To ensure a fair match up between the transformer model and the other XGBoost machines it will only look at one coin at a time, but that still makes the task multivariate because it ingests a whole ROW.

In [7]:
# Get a unique list of all timestamps and all 21 symbols
# all_times = master_df['open_time'].unique()
# all_symbols = master_df['symbol'].unique()

# # Create a "Perfectly Square" MultiIndex
# # This generates every possible combination of time and symbol
# full_index = pd.MultiIndex.from_product(
#     [all_times, all_symbols], 
#     names=['open_time', 'symbol']
# )

# # Apply this index to your DataFrame
# # Any coin that didn't exist at a specific timestamp will now get a synthetic row filled with NaNs
# master_df = master_df.set_index(['open_time', 'symbol']).reindex(full_index).reset_index()

# # Re-sort just to be absolutely sure of the chronological layout
# master_df = master_df.sort_values(by=['open_time', 'symbol']).reset_index(drop=True)

Inspect new master dataframe with NaN samples.

In [8]:
print(master_df.head(22))

                   open_time     open     high      low    close      volume  \
0  2017-08-17 04:00:00+00:00  4261.48  4313.62  4261.32  4308.83   47.181009   
1  2017-08-17 04:00:00+00:00   301.13   302.57   298.00   301.61  125.668770   
2  2017-08-17 05:00:00+00:00  4308.83  4328.69  4291.37  4315.32   23.234916   
3  2017-08-17 05:00:00+00:00   301.61   303.28   300.00   303.10  377.672460   
4  2017-08-17 06:00:00+00:00  4330.29  4345.45  4309.37  4324.35    7.229691   
5  2017-08-17 06:00:00+00:00   302.40   304.44   301.90   302.68  303.866720   
6  2017-08-17 07:00:00+00:00  4316.62  4349.99  4287.41  4349.99    4.443249   
7  2017-08-17 07:00:00+00:00   302.68   307.96   302.60   307.96  754.745100   
8  2017-08-17 08:00:00+00:00  4333.32  4377.85  4333.32  4360.69    0.972807   
9  2017-08-17 08:00:00+00:00   307.95   309.97   307.00   308.62  150.750290   
10 2017-08-17 09:00:00+00:00  4360.00  4445.78  4360.00  4444.00   10.763623   
11 2017-08-17 09:00:00+00:00   308.62   

Isolate identifiers for z-score normalization.

In [9]:
metadata_cols = ['open_time', 'symbol']
feature_cols = [col for col in master_df.columns if col not in metadata_cols]

Now we define the train, evaluation, and test splits. The data is split as follows:

- `train` : first candle -> 2025-12-30
- `evaluation` : 2025-12-31 -> 2026-03-30
- `test` : 2026-03-31 -> 2026-07-15

NOTE: Data ends at 2026-07-15 00:00 UTC, where that time is the exact time of the last hourly candle in the test set.

This drops December 31st 2025, and March 31st 2026 from all splits to ensure we don't accidentally perform data leakage across splits.

In [10]:
train_mask = master_df['open_time'] < '2025-12-31 00:00:00'
eval_mask  = (master_df['open_time'] >= '2026-01-01 00:00:00') & (master_df['open_time'] < '2026-03-31 00:00:00')
test_mask  = (master_df['open_time'] >= '2026-04-01 00:00:00') & (master_df['open_time'] < '2026-07-16 00:00:00')

Now we create a validity mask for NaNs in our dataframe that we ultimately want TS-JEPA to ignore. Remember market features do not turn on until 2017-12-12 where Binance has 4 coins (BTC, ETH, BNB, NEO) and can effectively begin capturing "market dynamics." HOWEVER, log_ret_126d does not turn on until 2017-12-21 @ 4:00 AM UTC for BTC & ETH. Therefore, the first effective sample that is NOT masked out would begin on that datetime.

In [11]:
master_df["valid_mask"] = (~master_df[feature_cols].isna()).all(axis=1)

In [12]:
pd.set_option('display.max_columns', None)

print(master_df.iloc[8060:8080])

                     open_time        open        high         low  \
8060 2017-12-21 02:00:00+00:00    307.9100    314.9900    305.0100   
8061 2017-12-21 02:00:00+00:00     75.7030     81.5000     74.9260   
8062 2017-12-21 03:00:00+00:00      5.3869      5.4900      5.2901   
8063 2017-12-21 03:00:00+00:00  16529.1300  16990.9900  16375.2900   
8064 2017-12-21 03:00:00+00:00    802.9100    818.0000    795.0000   
8065 2017-12-21 03:00:00+00:00    310.3000    319.4900    307.0000   
8066 2017-12-21 03:00:00+00:00     77.0000     80.0000     75.9080   
8067 2017-12-21 04:00:00+00:00      5.4900      5.4900      5.3400   
8068 2017-12-21 04:00:00+00:00  16901.0300  17050.0000  16808.5000   
8069 2017-12-21 04:00:00+00:00    816.0000    833.1000    806.0000   
8070 2017-12-21 04:00:00+00:00    317.0000    319.4200    311.5100   
8071 2017-12-21 04:00:00+00:00     78.6780     79.6450     76.5710   
8072 2017-12-21 05:00:00+00:00      5.4100      5.4220      5.3061   
8073 2017-12-21 05:0

According to [Joint-Embedding Predictive Learning of Latent Market States in U.S. Equities](https://openreview.net/pdf?id=BZfkxSasd3), Appendix section A, subsection A.1, the train/validation/test splits should be normalized within respect to the training split ONLY. The following code initiates the scalar on the training set, and is then applied statically to the evaluation, and test sets.

In [13]:
valid_mask = master_df['valid_mask'] == True

train_valid_mask = train_mask & valid_mask
eval_valid_mask  = eval_mask & valid_mask
test_valid_mask  = test_mask & valid_mask

# Initialize and Fit the Scaler ONLY on the Training Split
scaler = StandardScaler()
scaler.fit(master_df.loc[train_valid_mask, feature_cols])

# Transform All Splits using the Training Distribution
master_df.loc[train_valid_mask, feature_cols] = scaler.transform(master_df.loc[train_valid_mask, feature_cols])
master_df.loc[eval_valid_mask, feature_cols]  = scaler.transform(master_df.loc[eval_valid_mask, feature_cols])
master_df.loc[test_valid_mask, feature_cols]  = scaler.transform(master_df.loc[test_valid_mask, feature_cols])

Now we fill the NaNs in the master df with 0s to ensure we don't feed NaNs into JEPA. However, our validity dataframe allows us to know when a zero is a "true zero" (one served by the Binance api) or a "fake zero" (one we made up to keep shape consistency). "fake zeros" will be effectively blocked out from the attention mask in JEPA.

In [14]:
# master_df[feature_cols] = master_df[feature_cols].fillna(0.0)

Ensure there are zero NaNs left in our master df.

In [15]:
print(master_df.isna().sum())

open_time                0
open                     0
high                     0
low                      0
close                    0
volume                   0
symbol                   0
log_ret_1d             504
log_ret_5d            2520
log_ret_21d          10584
log_ret_63d          31752
log_ret_126d         63504
intraday_ret             0
mom_6_1              63504
hl_log_range             0
body_to_range            0
upper_shadow             0
lower_shadow             0
rvol_10               5040
rvol_21              10584
rvol_63              31752
rvol_ratio           31752
ewma_vol_hl10         5040
ewma_vol_hl20        10080
log_volume               0
log_dollar_volume        0
rel_dvol_21          10563
dvol_z_21            10563
amihud_illiq_1         504
amihud_illiq_21      11067
mkt_log_ret_1         6994
ret_vs_mkt_1          7402
mkt_rvol_21           6994
rel_vol              15562
valid_mask               0
dtype: int64


No NaNs left. 🎉 Let's inspect the scaled values first before exporting.

In [16]:
print(master_df.head(22))

                   open_time     open     high      low    close      volume  \
0  2017-08-17 04:00:00+00:00  4261.48  4313.62  4261.32  4308.83   47.181009   
1  2017-08-17 04:00:00+00:00   301.13   302.57   298.00   301.61  125.668770   
2  2017-08-17 05:00:00+00:00  4308.83  4328.69  4291.37  4315.32   23.234916   
3  2017-08-17 05:00:00+00:00   301.61   303.28   300.00   303.10  377.672460   
4  2017-08-17 06:00:00+00:00  4330.29  4345.45  4309.37  4324.35    7.229691   
5  2017-08-17 06:00:00+00:00   302.40   304.44   301.90   302.68  303.866720   
6  2017-08-17 07:00:00+00:00  4316.62  4349.99  4287.41  4349.99    4.443249   
7  2017-08-17 07:00:00+00:00   302.68   307.96   302.60   307.96  754.745100   
8  2017-08-17 08:00:00+00:00  4333.32  4377.85  4333.32  4360.69    0.972807   
9  2017-08-17 08:00:00+00:00   307.95   309.97   307.00   308.62  150.750290   
10 2017-08-17 09:00:00+00:00  4360.00  4445.78  4360.00  4444.00   10.763623   
11 2017-08-17 09:00:00+00:00   308.62   

Ensure scaling was done correctly.

In [17]:
# Filter to just the training split
train_df = master_df[train_mask]
train_validity = master_df[train_valid_mask]

# Pick a feature to test (e.g., 'open')
feature_to_test = 'open'

# print(train_validity.head(10))

# Extract ONLY the rows where the validity mask is 1 (real data)
real_data = train_df.loc[train_df['valid_mask'] == True, feature_to_test]

# Check the statistics
print(f"Mean: {real_data.mean():.6f}")
print(f"Std Dev: {real_data.std():.6f}")

# print(real_data.head(10))

Mean: 0.000000
Std Dev: 1.000000


In [18]:
print(train_df.head(10))

                  open_time     open     high      low    close      volume  \
0 2017-08-17 04:00:00+00:00  4261.48  4313.62  4261.32  4308.83   47.181009   
1 2017-08-17 04:00:00+00:00   301.13   302.57   298.00   301.61  125.668770   
2 2017-08-17 05:00:00+00:00  4308.83  4328.69  4291.37  4315.32   23.234916   
3 2017-08-17 05:00:00+00:00   301.61   303.28   300.00   303.10  377.672460   
4 2017-08-17 06:00:00+00:00  4330.29  4345.45  4309.37  4324.35    7.229691   
5 2017-08-17 06:00:00+00:00   302.40   304.44   301.90   302.68  303.866720   
6 2017-08-17 07:00:00+00:00  4316.62  4349.99  4287.41  4349.99    4.443249   
7 2017-08-17 07:00:00+00:00   302.68   307.96   302.60   307.96  754.745100   
8 2017-08-17 08:00:00+00:00  4333.32  4377.85  4333.32  4360.69    0.972807   
9 2017-08-17 08:00:00+00:00   307.95   309.97   307.00   308.62  150.750290   

    symbol  log_ret_1d  log_ret_5d  log_ret_21d  log_ret_63d  log_ret_126d  \
0  BTCUSDT         NaN         NaN          NaN     

Now export the data. At this point we switch to parquet, because these will be the final artifacts fed into our models.

In [19]:
export_dir = "normalized-features"

# make directory if it doesn't exist
os.makedirs(export_dir, exist_ok=True)

# export training dataset and validity mask to parquet
train_df.to_parquet(os.path.join(export_dir, "train_df.parquet"), index=False)

# export evaluation dataset and validity mask to parquet
eval_df = master_df[eval_mask]

eval_df.to_parquet(os.path.join(export_dir, "eval_df.parquet"), index=False)

# export test dataset and validity mask to parquet
test_df = master_df[test_mask]

test_df.to_parquet(os.path.join(export_dir, "test_df.parquet"), index=False)